In [ ]:
import gzip
import shutil

import pandas as pd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds

In [ ]:
import gzip
import shutil

import pandas as pd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds


# Step 1: Decompress the .gz file
with gzip.open('chirps-v2.0.2026.01.tif.gz', 'rb') as f_in:
    with open('chirps-v2.0.2026.01.tif', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

# Step 2: Open and inspect
with rasterio.open('chirps-v2.0.2026.01.tif') as src:
    print(f"Shape: {src.height} x {src.width}")
    print(f"CRS: {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Resolution: {src.res}")
    print(f"NoData: {src.nodata}")

    rain = src.read(1)

    # Example: clip to Datania-ish region
    window = from_bounds(24, -18, 33, -8, src.transform)
    rain_clip = src.read(1, window=window)

    rain = rain_clip


# Step 3: Handle NoData (CHIRPS uses -9999)
rain_clean = np.where(rain == -9999, np.nan, rain)

print(f"\nValid pixels: {np.count_nonzero(~np.isnan(rain_clean)):,}")
print(f"Min rainfall: {np.nanmin(rain_clean):.1f} mm")
print(f"Max rainfall: {np.nanmax(rain_clean):.1f} mm")
print(f"Mean rainfall: {np.nanmean(rain_clean):.1f} mm")

# Step 4: Visualize (global)
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(rain_clean, cmap='Blues', vmin=0,
               vmax=np.nanpercentile(rain_clean, 95))
ax.set_title('CHIRPS Rainfall — January 2026 (mm)')
plt.colorbar(im, ax=ax, label='Precipitation (mm)')
plt.show()

In [ ]:
# Option A: Clip while reading using a window (by coordinates)
from rasterio.windows import from_bounds

with rasterio.open('chirps-v2.0.2026.01.tif') as src:
    # Example: clip to Datania-ish region

    rain_clip = src.read(1)

In [ ]:
import geopandas as gpd
from pathlib import Path

rootPath = Path().cwd().parent.parent
countries = gpd.read_file(rootPath / "data" / "geo" / "africa" / "africa.geo.json")

country_name = "Kenya"  # <-- change
country = countries.loc[countries["name"] == country_name]

In [ ]:
from rasterio.plot import show
from rasterio.plot import plotting_extent

SCALE_FACTOR = 10  # 10x smaller → ~5 km pixels instead of ~500 m

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

with rasterio.open('nighttime_lights_africa.tif') as src:
    band = src.read(1).astype('float32')
    nodata = src.nodata
    mask = (band == nodata) if nodata is not None else np.isnan(band)
    valid = band[~mask]

    print("min, max:", valid.min(), valid.max())
    p2, p98 = np.percentile(valid, (2, 98))
    print("2/98 percentiles:", p2, p98)



    # 1) rasterio.show default behavior
    show(src, ax=axes[0], cmap='magma', title='rasterio.show (default)')
    axes[0].axis('off')

    # 2) rasterio.show with percent_range (histogram stretch)
    show(src, ax=axes[1], cmap='magma', percent_range=(2, 98), title='rasterio.show (percent_range=(2,98))')
    axes[1].axis('off')

# 3) explicit linear mapping using computed percentiles
axes[2].imshow(band, cmap='magma', vmin=p2, vmax=p98)
axes[2].set_title('imshow with vmin/vmax = 2/98 percentiles')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
p2, p98

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

p2, p98 = np.percentile(lights, (2, 98))

ax.imshow(lights, extent=extent, cmap='magma', vmin=p2, vmax=p98)

In [ ]:
np.percentile(lights, (2, 100))

In [ ]:
import geopandas as gpd
from pathlib import Path

import gzip
import shutil
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.mask import mask
from rasterio.plot import plotting_extent


rootPath = Path().cwd().parent.parent
countries = gpd.read_file(rootPath / "data" / "geo" / "africa" / "africa.geo.json")



# 1. Decompress
AFRICA_BOUNDS = (-18, -35, 52, 38)  # (west, south, east, north)
SCALE_FACTOR = 10  # 10x smaller → ~5 km pixels instead of ~500 m


with rasterio.open('nighttime_lights_global.tif') as src:
    # malawi, zambia, south africa, tanzania, angola
    for country_name in ["Malawi", "Zambia", "South Africa", "Tanzania", "Angola"]:
        country = countries.loc[countries["name"] == country_name]

        country = country.to_crs(src.crs)

        # Mask/crop
        out_image, out_transform = mask(
            src,
            country.geometry,   # shapes
            crop=True,
            nodata=src.nodata,  # or set your own nodata, e.g. -9999
            all_touched=True   # True includes any pixel that touches polygon
        )

        # if mask returned a masked array, fill masked values with nodata
        if hasattr(out_image, "filled"):
            out_image = out_image.filled(src.nodata)

        # build metadata for output
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "count": out_image.shape[0],
            "dtype": out_image.dtype,
            "nodata": src.nodata,
            "compress": "deflate"  # optional, saves disk space
        })

        out_tif_path = f"{country_name}_lights.tif"
        with rasterio.open(out_tif_path, "w", **out_meta) as dest:
            dest.write(out_image)

        print(f"  wrote {out_tif_path}")



In [ ]:
with rasterio.open('chirps-v2.0.2026.01.tif') as src:
    for country_name in ["Malawi", "Zambia", "South Africa", "Tanzania", "Angola"]:
        country = countries.loc[countries["name"] == country_name]

        country = country.to_crs(src.crs)

        # Mask/crop
        out_image, out_transform = mask(
            src,
            country.geometry,   # shapes
            crop=True,
            nodata=src.nodata,  # or set your own nodata, e.g. -9999
            all_touched=True   # True includes any pixel that touches polygon
        )

        # if mask returned a masked array, fill masked values with nodata
        if hasattr(out_image, "filled"):
            out_image = out_image.filled(src.nodata)

        # build metadata for output
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "count": out_image.shape[0],
            "dtype": out_image.dtype,
            "nodata": src.nodata,
            "compress": "deflate"  # optional, saves disk space
        })

        out_tif_path = f"{country_name}_chirps.tif"
        with rasterio.open(out_tif_path, "w", **out_meta) as dest:
            dest.write(out_image)

        print(f"  wrote {out_tif_path}")

In [ ]:
# plot malawi chirps with country boundary
with rasterio.open('Malawi_lights.tif') as src:
    arr = src.read(1)
    extent = plotting_extent(src)

    country = countries[countries["name"] == "Malawi"].to_crs(src.crs)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(arr, extent=extent, cmap="Blues", vmin=0, vmax=np.nanpercentile(arr, 99))
country.boundary.plot(ax=ax, edgecolor="cyan", linewidth=2)
ax.set_title("Malawi - CHIRPS Rainfall (mm)")
plt.colorbar(im, ax=ax, label='Precipitation (mm)', shrink=0.6)
plt.show()




In [ ]:
vmax = np.nanpercentile(arr, 99)
fig, ax = plt.subplots(figsize=(80, 60))
ax.imshow(arr, extent=extent, cmap="magma", vmax=vmax )
country.boundary.plot(ax=ax, edgecolor="cyan", linewidth=2)
ax.set_title(country_name)
plt.show()

In [ ]:
del lights

In [ ]:
import gzip
import shutil
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling


# 1. Decompress
#with gzip.open('VNL_npp_2024_global_vcmslcfg_v2_c202502261200.average_masked.dat.tif.gz', 'rb') as f_in:
#    with open('nighttime_lights_global.tif', 'wb') as f_out:
#        shutil.copyfileobj(f_in, f_out)


import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 2. CLIP TO AFRICA
# ============================================================
# Africa approximate bounding box
AFRICA_BOUNDS = (-18, -35, 52, 38)  # (west, south, east, north)
SCALE_FACTOR = 10  # 10x smaller → ~5 km pixels instead of ~500 m


with rasterio.open('nighttime_lights_global.tif') as src:
    window = from_bounds(*AFRICA_BOUNDS, transform=src.transform)

    # Calculate reduced output size
    out_height = int(window.height // SCALE_FACTOR)
    out_width = int(window.width // SCALE_FACTOR)

    # Read directly at reduced resolution
    lights = src.read(
        1,
        window=window,
        out_shape=(out_height, out_width),
        resampling=Resampling.average   # average pixels, not just skip
    )

# ============================================================
# 3. CLEAN UP NODATA / ZEROS
# ============================================================
# In the "average-masked" product, background is 0 and nodata is
# typically 0 or -9999. Replace with NaN for clean stats & display.
lights_clean = lights.astype(float)
lights_clean[lights_clean <= 0] = np.nan



print(f"  Valid pixels : {np.count_nonzero(~np.isnan(lights_clean)):,}")
print(f"  Mean radiance: {np.nanmean(lights_clean):.2f} nW/cm²/sr")
print(f"  Max radiance : {np.nanmax(lights_clean):.1f} nW/cm²/sr")

# ============================================================
# 4. VISUALIZE
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Left panel: linear stretch clipped at 99th percentile ---
ax1 = axes[0]
vmax = np.nanpercentile(lights, 99)
im1 = ax1.imshow(
    lights,
    cmap='magma',
    vmin=0, vmax=vmax,
    extent=AFRICA_BOUNDS,          # (west, east, south, north) for imshow
    origin='upper'
)
ax1.set_title('Nighttime Lights — Africa\n(linear, clipped at 99th pctl)')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
plt.colorbar(im1, ax=ax1, label='Radiance (nW/cm²/sr)', shrink=0.6)

# --- Right panel: log scale to reveal dim areas ---
ax2 = axes[1]
lights_log = np.log1p(lights)   # log(1 + x), safe for zeros/NaN
im2 = ax2.imshow(
    lights_log,
    cmap='magma',
    vmin=0, vmax=np.nanpercentile(lights_log, 99),
    extent=AFRICA_BOUNDS,
    origin='upper'
)
ax2.set_title('Nighttime Lights — Africa\n(log scale — reveals rural detail)')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
plt.colorbar(im2, ax=ax2, label='log(1 + Radiance)', shrink=0.6)

plt.tight_layout()
plt.savefig('nighttime_lights_africa.png', dpi=200, bbox_inches='tight')
plt.show()

print("✓ Map saved: nighttime_lights_africa.png")

In [ ]:
import requests

file = "https://soskgoc08swc0gs4wocw8oso.rowsquared.org/api/public/dl/7H1r0mtr?inline=true"
r = requests.get(file)


In [ ]:
r.text